In [ ]:
import pandas as pd
import re
import concurrent.futures
from pathlib import Path
from typing import Any
import time
from datetime import datetime
from tqdm.auto import tqdm

# =====================================================================
# 1. KONFIGURASI DIREKTORI (LOCKED)
# =====================================================================
base_dir = Path("data_spliting")  # Direktori sumber
output_dir = Path("hasil_ekstraksi") # Direktori tujuan
output_dir.mkdir(parents=True, exist_ok=True)

if not base_dir.exists():
    raise FileNotFoundError(f"Folder sumber '{base_dir}' tidak ditemukan!")

LABEL_BENIGN    = 0
LABEL_MALICIOUS = 1
ENCODING_ORDER  = ["utf-8", "utf-16", "cp1252", "latin-1"]

# =====================================================================
# 2. SETUP FITUR 
# =====================================================================
_CHAR_COL = {
    "$": "char_dollar",    "(": "char_lparen",    ")": "char_rparen",
    "[": "char_lbracket",  "]": "char_rbracket",   "{": "char_lbrace",
    "}": "char_rbrace",    ".": "char_dot",        ",": "char_comma",
    "+": "char_plus",      "-": "char_minus",      "*": "char_star",
    "/": "char_slash",     "=": "char_equal",      "!": "char_exclaim",
    "%": "char_percent",   "^": "char_caret",      "&": "char_ampersand",
    "`": "char_backtick",
}

TARGET_KEYWORDS = [
    "Get-Command", "Get-Help", "Get-Process", "Get-Service", "Get-ChildItem", 
    "Get-WmiObject", "Get-Alias", "Get-Member", "Set-Location", "Get-Location", 
    "Get-Content", "Set-Content", "Add-Content", "Write-Host", "Write-Output", 
    "Write-Verbose", "Write-Error", "Start-Transcript", "Stop-Transcript", 
    "Set-StrictMode", "Protect-CmsMessage", "Unprotect-CmsMessage", 
    "Test-NetConnection", "Test-Path", "Import-Module", "Select-Object", 
    "Where-Object", "ForEach-Object", "Get-Acl", "Set-Acl", "Get-FileHash", 
    "pwd", "gl", "Invoke-Obfuscation", "Invoke-Mimikatz", "PowerSploit", 
    "PowerShell Empire", "Nishang", "PowerUp", "p0wnedShell", "PowerOPS", 
    "Poweliks", "Kovter", "Shellcode", "Start-Process", "Invoke-WebRequest", 
    "Invoke-RestMethod", "-f", "[Convert]::FromBase64String", 
    "[System.Convert]::FromBase64String", "[Convert]::ToBase64String", 
    "[System.Convert]::ToBase64String", "System.IO.Compression.DeflateStream", 
    "IO.MemoryStream", "[Runtime.InteropServices.Marshal]::PtrToStringAnsi", 
    "[Runtime.InteropServices.Marshal]::SecureStringToGlobalAllocAnsi", 
    "$True", "$False", "ConvertFrom-SecureString", "ConvertTo-SecureString",
    "Invoke-Expression", "IEX", "-EncodedCommand", "-exec bypass", 
    "-ExecutionPolicy Bypass", "System.Net.WebClient", "Net.WebClient", 
    "DownloadString", "DownloadFile", "New-Object"
]

# Tepat 11 AST sesuai kebutuhan murni
AST_TARGET_NODES = [
    "AssignmentStatementAst", "BinaryExpressionAst", "CommandAst", 
    "InvokeMemberExpressionAst", "PipelineAst", "StringConstantExpressionAst", 
    "TypeExpressionAst", "VariableExpressionAst", 
    "ConvertExpressionAst", "ArrayLiteralAst", "ParenExpressionAst"
]

# =====================================================================
# 3. FUNGSI EKSTRAKSI (TAHAP 1)
# =====================================================================
def _read_ps1(fp: Path) -> str:
    for enc in ENCODING_ORDER:
        try:
            return fp.read_text(encoding=enc)
        except (UnicodeDecodeError, OSError):
            continue
    return ""

def _stat_features(script: str) -> dict[str, Any]:
    lines = script.splitlines() or [""]
    lengths = [len(ln) for ln in lines]
    has_url_ip = int(bool(re.search(r"https?://", script, re.I) or re.search(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", script)))
    feats = {
        "script_length"  : len(script),
        "line_count"     : len(lines),
        "avg_line_length": round(sum(lengths) / len(lengths), 4) if lengths else 0,
        "max_line_length": max(lengths) if lengths else 0,
        "has_url_or_ip"  : has_url_ip,
    }
    for char, col in _CHAR_COL.items():
        feats[col] = script.count(char)
    return feats

def _lex_features(script: str) -> dict[str, Any]:
    feats = {
        "var_count" : len(re.findall(r"\$[a-zA-Z_]\w*", script)),
        "string_literal_count": len(re.findall(r"'[^']*'", script)) + len(re.findall(r'"[^"]*"', script)),
    }
    sl = script.lower()
    for kw in TARGET_KEYWORDS:
        col = "kw_" + re.sub(r"[^a-z0-9]", "_", kw.lower())
        feats[col] = len(re.findall(re.escape(kw.lower()), sl))
    return feats

def create_base_csv(target_folder: Path, output_csv: Path, desc: str):
    all_files = []
    benign_path = target_folder / "benign"
    malicious_path = target_folder / "malicious"
    
    if benign_path.exists():
        all_files.extend((fp, LABEL_BENIGN) for fp in sorted(benign_path.glob("*.ps1")))
    if malicious_path.exists():
        all_files.extend((fp, LABEL_MALICIOUS) for fp in sorted(malicious_path.glob("*.ps1")))

    if not all_files:
        return False

    records = []
    def _process_file(fp, lbl):
        script = _read_ps1(fp)
        row = {"filename": fp.name, "label": lbl, **_stat_features(script), **_lex_features(script)}
        # Menyiapkan kolom kosong khusus untuk 11 AST
        row.update({n: 0 for n in AST_TARGET_NODES})
        return row

    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = [executor.submit(_process_file, fp, lbl) for fp, lbl in all_files]
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(all_files), desc=f"Stat/Lex ({desc})"):
            records.append(future.result())

    df = pd.DataFrame(records).fillna(0)
    df.to_csv(output_csv, index=False)
    print(f"     [OK] CSV Dasar Dibuat: {output_csv.name} ({len(df)} baris)")
    return True

# =====================================================================
# 4. EKSEKUSI TAHAP 1
# =====================================================================
print("="*80)
print(f"▶ MULAI TAHAP 1 (EKSTRAKSI STATISTIK & LEKSIKAL): {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)
waktu_mulai = time.time()

kategori_bersih = ["train_set", "test_set"]
kategori_obfuskasi = ["train_set_obf", "test_set_obf"]

# 1A. Kategori Bersih
for set_name in kategori_bersih:
    folder_target = base_dir / set_name
    if folder_target.exists():
        csv_output = output_dir / f"fitur_{set_name}.csv"
        create_base_csv(folder_target, csv_output, set_name)

# 1B. Kategori Obfuskasi
for set_name in kategori_obfuskasi:
    folder_set_obf = base_dir / set_name
    if folder_set_obf.exists():
        teknik_folders = [f for f in folder_set_obf.iterdir() if f.is_dir()]
        for teknik_dir in teknik_folders:
            teknik_name = teknik_dir.name
            csv_output = output_dir / f"fitur_{set_name}_{teknik_name}.csv"
            create_base_csv(teknik_dir, csv_output, f"{set_name}/{teknik_name}")

durasi = (time.time() - waktu_mulai) / 60
print("="*80)
print(f"⏹ TAHAP 1 SELESAI. Total Waktu: {durasi:.2f} Menit.")
print("="*80)

► MULAI TAHAP 1 (EKSTRAKSI STATISTIK & LEKSIKAL): 2026-07-13 07:54:15
Stat/Lex (train_set): 100%|██████████| 4949/4949 [00:13<00:00, 378.85it/s]
    [OK] CSV Dasar Dibuat: fitur_train_set.csv (4949 baris)
Stat/Lex (test_set): 100%|██████████| 1238/1238 [00:04<00:00, 308.13it/s]
    [OK] CSV Dasar Dibuat: fitur_test_set.csv (1238 baris)
Stat/Lex (train_set_obf/ASCII_Encoding): 100%|██████████| 330/330 [00:00<00:00, 515.00it/s]
    [OK] CSV Dasar Dibuat: fitur_train_set_obf_ASCII_Encoding.csv (330 baris)
Stat/Lex (train_set_obf/String_Concatenation): 100%|██████████| 330/330 [00:00<00:00, 552.08it/s]
    [OK] CSV Dasar Dibuat: fitur_train_set_obf_String_Concatenation.csv (330 baris)
Stat/Lex (train_set_obf/String_Reordering): 100%|██████████| 330/330 [00:00<00:00, 534.51it/s]
    [OK] CSV Dasar Dibuat: fitur_train_set_obf_String_Reordering.csv (330 baris)
Stat/Lex (train_set_obf/Token_Manipulation): 100%|██████████| 330/330 [00:00<00:00, 643.14it/s]
    [OK] CSV Dasar Dibuat: fitur_train

In [ ]:
import subprocess
import json
import pandas as pd
import concurrent.futures
from pathlib import Path
import time
from datetime import datetime
from tqdm.auto import tqdm

# =====================================================================
# 1. FUNGSI PENAMBALAN AST INLINE (100% KOMPATIBEL & ANTI-HALU)
# =====================================================================
AST_TARGET_NODES = [
    "AssignmentStatementAst", "BinaryExpressionAst", "CommandAst", 
    "InvokeMemberExpressionAst", "PipelineAst", "StringConstantExpressionAst", 
    "TypeExpressionAst", "VariableExpressionAst", 
    "ConvertExpressionAst", "ArrayLiteralAst", "ParenExpressionAst"
]
ps_ast_array = ", ".join(f"'{node}'" for node in AST_TARGET_NODES)

def get_ast_inline(file_path: Path) -> dict:
    ps_script = f"""
    $targets = @({ps_ast_array})
    $path = '{file_path.resolve()}'
    try {{
        $tok = $null; $err = $null
        $ast = [System.Management.Automation.Language.Parser]::ParseFile($path, [ref]$tok, [ref]$err)
        
        $counts = @{{}}
        foreach ($t in $targets) {{ $counts[$t] = 0 }}
        
        # MENGGUNAKAN FINDALL MASSAL (Sangat cepat dan kompatibel di semua versi Windows)
        $allNodes = $ast.FindAll({{$true}}, $true)
        
        foreach ($node in $allNodes) {{
            $name = $node.GetType().Name
            if ($counts.ContainsKey($name)) {{
                $counts[$name]++
            }}
        }}

        $counts | ConvertTo-Json -Compress
    }} catch {{
        $fb = @{{}}
        foreach ($t in $targets) {{ $fb[$t] = 0 }}
        $fb | ConvertTo-Json -Compress
    }}
    """
    try:
        res = subprocess.run(
            ["pwsh", "-NoProfile", "-NonInteractive", "-Command", "-"],
            input=ps_script, capture_output=True, text=True, timeout=25
        )
        if res.returncode == 0 and res.stdout.strip():
            raw = json.loads(res.stdout.strip())
            # Mengembalikan persis 11 fitur AST (tanpa ast_depth)
            return {n: int(raw.get(n, 0)) for n in AST_TARGET_NODES}
    except Exception:
        return None
    return None

def _process_ast_row(args):
    idx, filename, label, target_folder = args
    subfolder = "malicious" if label == 1 else "benign"
    target_file = target_folder / subfolder / filename
    
    ast_result = get_ast_inline(target_file) if target_file.exists() else None
    return idx, ast_result

def append_ast_to_csv(csv_path: Path, target_folder: Path, desc: str):
    if not csv_path.exists():
        return
        
    df = pd.read_csv(csv_path)
    tasks = [(idx, row['filename'], row['label'], target_folder) for idx, row in df.iterrows()]

    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        futures = [executor.submit(_process_ast_row, task) for task in tasks]
        
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(tasks), desc=f"AST Patch ({desc})"):
            idx, ast_result = future.result()
            if ast_result is not None:
                for col_name, value in ast_result.items():
                    df.at[idx, col_name] = value

    df.to_csv(csv_path, index=False)
    print(f"     [OK] Fitur AST Ditambahkan: {csv_path.name}")

# =====================================================================
# 2. EKSEKUSI TAHAP 2 (AST PATCHING)
# =====================================================================
print("="*80)
print(f"▶ MULAI TAHAP 2 (PENAMBALAN FITUR AST): {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)
waktu_mulai_ast = time.time()

base_dir = Path("data_spliting")
output_dir = Path("hasil_ekstraksi")
kategori_bersih = ["train_set", "test_set"]
kategori_obfuskasi = ["train_set_obf", "test_set_obf"]

# 2A. Kategori Bersih
for set_name in kategori_bersih:
    folder_target = base_dir / set_name
    if folder_target.exists():
        csv_output = output_dir / f"fitur_{set_name}.csv"
        append_ast_to_csv(csv_output, folder_target, set_name)

# 2B. Kategori Obfuskasi
for set_name in kategori_obfuskasi:
    folder_set_obf = base_dir / set_name
    if folder_set_obf.exists():
        teknik_folders = [f for f in folder_set_obf.iterdir() if f.is_dir()]
        for teknik_dir in teknik_folders:
            teknik_name = teknik_dir.name
            csv_output = output_dir / f"fitur_{set_name}_{teknik_name}.csv"
            append_ast_to_csv(csv_output, teknik_dir, f"{set_name}/{teknik_name}")

durasi_ast = (time.time() - waktu_mulai_ast) / 60
print("="*80)
print(f"⏹ TAHAP 2 SELESAI. Total Waktu AST: {durasi_ast:.2f} Menit.")
print("="*80)

► MULAI TAHAP 2 (PENAMBALAN FITUR AST): 2026-07-13 07:55:04
AST Patch (train_set): 100%|██████████| 4949/4949 [18:11<00:00, 4.53it/s]
    [OK] Fitur AST Ditambahkan: fitur_train_set.csv
AST Patch (test_set): 100%|██████████| 1238/1238 [04:43<00:00, 4.37it/s]
    [OK] Fitur AST Ditambahkan: fitur_test_set.csv
AST Patch (train_set_obf/ASCII_Encoding): 100%|██████████| 330/330 [01:13<00:00, 4.54it/s]
    [OK] Fitur AST Ditambahkan: fitur_train_set_obf_ASCII_Encoding.csv
AST Patch (train_set_obf/String_Concatenation): 100%|██████████| 330/330 [01:13<00:00, 4.51it/s]
    [OK] Fitur AST Ditambahkan: fitur_train_set_obf_String_Concatenation.csv
AST Patch (train_set_obf/String_Reordering): 100%|██████████| 330/330 [01:16<00:00, 4.32it/s]
    [OK] Fitur AST Ditambahkan: fitur_train_set_obf_String_Reordering.csv
AST Patch (train_set_obf/Token_Manipulation): 100%|██████████| 330/330 [01:10<00:00, 4.72it/s]
    [OK] Fitur AST Ditambahkan: fitur_train_set_obf_Token_Manipulation.csv
AST Patch (test_

In [3]:
import pandas as pd
from pathlib import Path

# =====================================================================
# 1. KONFIGURASI DIREKTORI & URUTAN TEKNIK
# =====================================================================
input_dir = Path("hasil_ekstraksi")
output_file = input_dir / "fitur_train_set_obf.csv"

# URUTAN TEKNIK (Dikunci sesuai permintaan Anda)
# Pastikan nama di bawah ini sama persis dengan akhiran nama file CSV Anda
teknik_list = [
    "ASCII_Encoding",
    "String_Concatenation",
    "String_Reordering",
    "Token_Manipulation"
]

all_dataframes = []
TARGET_PER_TEKNIK = 375

print("="*80)
print("▶ MULAI PENGGABUNGAN DATA (TRAIN SET OBFUSCATION)")
print("="*80)

# =====================================================================
# 2. PROSES FILTERING DAN PEMOTONGAN TEPAT 375
# =====================================================================
for tech in teknik_list:
    file_path = input_dir / f"fitur_train_set_obf_{tech}.csv"
    
    if file_path.exists():
        df = pd.read_csv(file_path)
        
        # Tambahkan label teknik
        df.insert(1, 'teknik_obfuskasi', tech)
        
        # Identifikasi kolom fitur 
        kolom_fitur = [col for col in df.columns if col not in ['filename', 'teknik_obfuskasi', 'label']]
        
        # 1. Buang data sampah (kosong semua)
        mask_semua_nol = (df[kolom_fitur] == 0).all(axis=1)
        df_bersih = df[~mask_semua_nol].copy()
        
        # 2. POTONG PAS 375 BARIS (Kunci Utama)
        df_terpilih = df_bersih.head(TARGET_PER_TEKNIK)
        
        print(f"[*] Memproses : {tech}")
        print(f"    - Tersedia (bersih) : {len(df_bersih)} baris")
        print(f"    - Diambil & Masuk   : {len(df_terpilih)} baris\n")
        
        all_dataframes.append(df_terpilih)
    else:
        print(f"[!] PERINGATAN: File tidak ditemukan -> {file_path.name}")

# =====================================================================
# 3. CETAK CSV DAN LAPORAN (HARUS 1500)
# =====================================================================
if all_dataframes:
    # Gabungkan menjadi 1 DataFrame
    df_final = pd.concat(all_dataframes, ignore_index=True)
    
    # Simpan menjadi CSV baru
    df_final.to_csv(output_file, index=False)
    
    print("="*80)
    print(f"✅ PENGGABUNGAN SELESAI! File tersimpan: {output_file.name}")
    print(f"✅ Total Baris Keseluruhan: {len(df_final)}")
    print("\n📊 PETA URUTAN BARIS ABSOLUT:")
    
    start_idx = 1
    for tech, df_part in df_final.groupby('teknik_obfuskasi', sort=False):
        count = len(df_part)
        end_idx = start_idx + count - 1
        print(f" - {tech:<25} : Baris ke-{start_idx} hingga {end_idx}")
        start_idx = end_idx + 1
        
    print("="*80)
else:
    print("\n❌ Gagal menggabungkan: Tidak ada file CSV yang ditemukan.")

▶ MULAI PENGGABUNGAN DATA (TRAIN SET OBFUSCATION)
[*] Memproses : ASCII_Encoding
    - Tersedia (bersih) : 1350 baris
    - Diambil & Masuk   : 375 baris

[*] Memproses : String_Concatenation
    - Tersedia (bersih) : 1350 baris
    - Diambil & Masuk   : 375 baris

[*] Memproses : String_Reordering
    - Tersedia (bersih) : 1350 baris
    - Diambil & Masuk   : 375 baris

[*] Memproses : Token_Manipulation
    - Tersedia (bersih) : 1350 baris
    - Diambil & Masuk   : 375 baris

✅ PENGGABUNGAN SELESAI! File tersimpan: fitur_train_set_obf.csv
✅ Total Baris Keseluruhan: 1500

📊 PETA URUTAN BARIS ABSOLUT:
 - ASCII_Encoding            : Baris ke-1 hingga 375
 - String_Concatenation      : Baris ke-376 hingga 750
 - String_Reordering         : Baris ke-751 hingga 1125
 - Token_Manipulation        : Baris ke-1126 hingga 1500


C:\Users\ASUS\AppData\Local\Temp\ipykernel_28844\679966411.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(1, 'teknik_obfuskasi', tech)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_28844\679966411.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(1, 'teknik_obfuskasi', tech)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_28844\679966411.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining al